<a href="https://colab.research.google.com/github/anjaliborkar150-droid/BuildingAiAgent/blob/main/Building_AI_Agent001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qU langchain

In [2]:
!pip show langchain

Name: langchain
Version: 1.2.15
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [3]:
!pip install -qU langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 18.8 MB/s eta 0:00:00


In [4]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

In [6]:
gemini_api_key = userdata.get("GEMINI_API_KEY")

In [11]:
model = init_chat_model(
    model="google_genai:gemini-2.5-flash",
    api_key = gemini_api_key)

Create Skill demand tool

In [12]:
!pip install -qU langchain-tavily

In [15]:
from langchain_tavily import TavilySearch
from pprint import pprint

In [16]:
tavily_api_key = userdata.get("TAVILY_API_KEY")

In [20]:
skill_demand_tool = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced",
    tavily_api_key=tavily_api_key

)

Create Job search tool

In [21]:
rapid_api_key = userdata.get("RAPID_API_KEY")

In [22]:
import requests
from langchain.tools import tool

In [38]:
import requests
from langchain.tools import tool
from google.colab import userdata

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
    print(f"\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")

    rapidapi_key = userdata.get('RAPID_API_KEY')

    url = "https://jsearch.p.rapidapi.com/search"
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    querystring = {
        "query": f"{skill} in {location}",
        "page": "1",
        "country": "in",
        "employment_types": "INTERN,FULLTIME",
        "job_requirements": "no_experience,under_3_years_experience"
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    jobs = data.get("data", [])
    print(f"Found {len(jobs)} jobs\n")

    result = []
    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link")
        })
    return result

Creating an agent

In [39]:
from langchain.agents import create_agent

In [40]:
system_prompt= """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

In [41]:
agent = create_agent(
    model= model,
    system_prompt= system_prompt,
    tools=[skill_demand_tool, search_jobs],
    debug = True
)

In [42]:
user_query= "Whats the demand for AI?, show me related jobs in India"

In [43]:
response = agent.invoke({
    "messages":[
        {
          "role":"user",
          "content": user_query

          }
        ]
})

[values] {'messages': [HumanMessage(content='Whats the demand for AI?, show me related jobs in India', additional_kwargs={}, response_metadata={}, id='a63bc642-d47a-462e-8d31-323a90ce1311')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "AI skill demand and salary trends"}'}, '__gemini_function_call_thought_signatures__': {'e5339ead-3669-4624-8009-d5ebbc8fc9e7': 'CvQGAQw51sc1YANtVe9lxfFsit+DAyIhR7KQd7Z82f/3M9DYIYArYszAHj0N2NYtFj8VaVgfNDfqikWeckajPSJ4OBCrAWyTF6YEK1vGIgMU7JdDfUm2s+gqe592VLC2npbc29npsbjiIif7bGqjkHT0d7AuUqU4aWxEg7wVeJ4QMe3G4DFshQbp8W7vOsBMaoX5mmxeUlcosnUoKIZW7dfFm1nev5p4GSMkzxlKg/irEfxwgEPTgqMqt7YYUdX65Ipzr4iHAw5SSreZn3iBG5HuFmZbzNTezvrzDl8oMeLRz8T4KG2WfUyiFU/b6YzyGQ+iJpxgCoXL0JkTZaqnsNoZmfmkqvz9GRawKnVdLV7V3OfbQf1MesGhBwEv2LjU/GChmM44F18LCOXBmmkyodC/5XJGX6r9vhMRoYY9RmsjMYkFpRB/gb6HJWmJwfrYDeL8oyWjMwmVwiyGMdGWv4okmxGYLVxa8fpBJH+BYHHS2/hUvVmSregH2Z4vzT3EuYqbgogBs94rBP3juD9p/gHuT